# Cell 1 -- Installing Required Libraries

Before we begin, we need to install a few external Python libraries that are not available by default in the Kaggle environment.

- **transformers**: Provides access to pretrained deep learning models like HuBERT, which is a pretrained audio Transformer used for audio classification tasks.
- **wandb** (Weights and Biases): A tool for experiment tracking. It logs all our training metrics (loss, accuracy, F1 score, learning rate) so we can compare models easily on a dashboard.
- **librosa**: A popular Python library for audio analysis. We use it to extract handcrafted audio features like MFCCs, Chroma, and Spectral Contrast for our XGBoost baseline.
- **xgboost**: A powerful gradient-boosted tree algorithm used as our traditional ML baseline model.

The `-q` flag keeps the installation output quiet.


# Cell 1 -- Installing Required Libraries

Before we begin, we need to install a few external Python libraries that are not available by default in the Kaggle environment.

- **transformers**: Provides access to pretrained deep learning models like HuBERT, which is a pretrained audio Transformer used for audio classification tasks.
- **wandb** (Weights and Biases): A tool for experiment tracking. It logs all our training metrics (loss, accuracy, F1 score, learning rate) so we can compare models easily on a dashboard.
- **librosa**: A popular Python library for audio analysis. We use it to extract handcrafted audio features like MFCCs, Chroma, and Spectral Contrast for our XGBoost baseline.
- **xgboost**: A powerful gradient-boosted tree algorithm used as our traditional ML baseline model.

The `-q` flag keeps the installation output quiet.


In [1]:
# ============================================================
# Cell 1: Installs (Run on CPU - no GPU needed)
# ============================================================
!pip install transformers wandb librosa xgboost -q

# Cell 2 -- Imports, Seed, and Configuration

This cell does three important things:

**1. Imports:** We import all the libraries we will use throughout this notebook, including PyTorch (for building neural networks), torchaudio (for audio loading and transformations), scikit-learn (for evaluation metrics), and others.

**2. Reproducibility (Seed):** We set `SEED = 42` and apply it to Python's `random`, NumPy, and PyTorch. This ensures that every time we run the notebook, the random operations (like shuffling data or initializing weights) produce the same results, making our experiments reproducible.

**3. Constants:**
- `SAMPLE_RATE = 16000`: All audio is resampled to 16kHz. This is the standard rate for speech/audio models like HuBERT.
- `DURATION = 10`: We process 10-second audio clips. Test mashups are 30 seconds long, but our models are trained on 10-second segments.
- `NUM_CLASSES = 10`: We have 10 music genres to classify: blues, classical, country, disco, hiphop, jazz, metal, pop, reggae, rock.
- `genre_to_idx` / `idx_to_genre`: Dictionaries to convert between genre names and numeric labels (required by PyTorch).


# Cell 2 -- Imports, Seed, and Configuration

This cell does three important things:

**1. Imports:** We import all the libraries we will use throughout this notebook, including PyTorch (for building neural networks), torchaudio (for audio loading and transformations), scikit-learn (for evaluation metrics), and others.

**2. Reproducibility (Seed):** We set `SEED = 42` and apply it to Python's `random`, NumPy, and PyTorch. This ensures that every time we run the notebook, the random operations (like shuffling data or initializing weights) produce the same results, making our experiments reproducible.

**3. Constants:**
- `SAMPLE_RATE = 16000`: All audio is resampled to 16kHz. This is the standard rate for speech/audio models like HuBERT.
- `DURATION = 10`: We process 10-second audio clips. Test mashups are 30 seconds long, but our models are trained on 10-second segments.
- `NUM_CLASSES = 10`: We have 10 music genres to classify: blues, classical, country, disco, hiphop, jazz, metal, pop, reggae, rock.
- `genre_to_idx` / `idx_to_genre`: Dictionaries to convert between genre names and numeric labels (required by PyTorch).


In [2]:
# ============================================================
# Cell 2: Imports + Seed + Config
# ============================================================
import os
import random
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split

import librosa
import librosa.display
import IPython.display as ipd

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report

import wandb
from transformers import HubertForSequenceClassification
from tqdm import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Constants
SAMPLE_RATE = 16000
DURATION = 10         # seconds (increased from 5 for better genre recognition)
NUM_SAMPLES = SAMPLE_RATE * DURATION
NUM_CLASSES = 10

genres = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']
genre_to_idx = {g: i for i, g in enumerate(genres)}
idx_to_genre = {i: g for i, g in enumerate(genres)}

print("All imports done.")
print(f"PyTorch: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

All imports done.
PyTorch: 2.8.0+cu126
GPU Available: True
GPU Name: Tesla T4


# Cell 3 -- Path Configuration and Directory Exploration

Here we set up all the file paths for our dataset and explore its structure.

**Key directories:**
- `STEMS_DIR`: Contains the training data. Each genre has 100 songs, and each song has 4 stems: `vocals.wav`, `drums.wav`, `bass.wav`, `other.wav`.
- `MASHUPS_DIR`: Contains 3020 test audio files (mashups). These are noisy mixes of stems from different songs of the same genre, with added environmental noise.
- `NOISE_DIR`: Points to the ESC-50 dataset (2000 environmental sound clips). We use this to inject realistic background noise into our training data to make our model robust to the noisy test conditions.

**Important:** The test mashups are created by mixing stems from *different* songs of the same genre, not from the same song. This insight is critical and it motivates our Cross-Song Stem Mixing augmentation strategy in later cells.


# Cell 3 -- Path Configuration and Directory Exploration

Here we set up all the file paths for our dataset and explore its structure.

**Key directories:**
- `STEMS_DIR`: Contains the training data. Each genre has 100 songs, and each song has 4 stems: `vocals.wav`, `drums.wav`, `bass.wav`, `other.wav`.
- `MASHUPS_DIR`: Contains 3020 test audio files (mashups). These are noisy mixes of stems from different songs of the same genre, with added environmental noise.
- `NOISE_DIR`: Points to the ESC-50 dataset (2000 environmental sound clips). We use this to inject realistic background noise into our training data to make our model robust to the noisy test conditions.

**Important:** The test mashups are created by mixing stems from *different* songs of the same genre, not from the same song. This insight is critical and it motivates our Cross-Song Stem Mixing augmentation strategy in later cells.


In [3]:
# ============================================================
# Cell 3: Path Config + Directory Exploration
# CRITICAL: This cell discovers the actual file naming format
# ============================================================
BASE_DIR = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'

# Fallback for older competition path structure
if not os.path.exists(BASE_DIR):
    BASE_DIR = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'

STEMS_DIR = os.path.join(BASE_DIR, 'genres_stems')
MASHUPS_DIR = os.path.join(BASE_DIR, 'mashups')

# Auto-detect noise folder (ESC-50-master vs ESC-50)
if os.path.exists(os.path.join(BASE_DIR, 'ESC-50-master', 'audio')):
    NOISE_DIR = os.path.join(BASE_DIR, 'ESC-50-master', 'audio')
elif os.path.exists(os.path.join(BASE_DIR, 'ESC-50', 'audio')):
    NOISE_DIR = os.path.join(BASE_DIR, 'ESC-50', 'audio')
else:
    NOISE_DIR = None
    print("WARNING: Noise directory not found!")

print(f"BASE_DIR:   {BASE_DIR}")
print(f"STEMS_DIR:  {STEMS_DIR}")
print(f"MASHUPS_DIR:{MASHUPS_DIR}")
print(f"NOISE_DIR:  {NOISE_DIR}")
print()

# --- Explore stems directory ---
print("=" * 50)
print("TRAINING DATA (Stems per Genre)")
print("=" * 50)
for genre in genres:
    genre_path = os.path.join(STEMS_DIR, genre)
    if os.path.exists(genre_path):
        songs = sorted(os.listdir(genre_path))
        # Check what stems exist in first song
        first_song_stems = os.listdir(os.path.join(genre_path, songs[0])) if songs else []
        print(f"  {genre:12s}: {len(songs)} songs | stems: {first_song_stems}")

# --- CRITICAL: Explore mashup files to discover naming format ---
print()
print("=" * 50)
print("TEST DATA (Mashup Files)")
print("=" * 50)
mashup_files = sorted(os.listdir(MASHUPS_DIR))
print(f"Total mashup files: {len(mashup_files)}")
print(f"First 10 files: {mashup_files[:10]}")
print(f"Last 5 files:   {mashup_files[-5:]}")

# --- Check test.csv ID format ---
print()
print("=" * 50)
print("test.csv and sample_submission.csv")
print("=" * 50)
test_csv_path = os.path.join(BASE_DIR, 'test.csv')
sample_sub_path = os.path.join(BASE_DIR, 'sample_submission.csv')

test_df = pd.read_csv(test_csv_path)
print(f"test.csv shape: {test_df.shape}")
print(f"test.csv columns: {list(test_df.columns)}")
print(f"First 5 rows:\n{test_df.head()}")
print(f"\nID dtype: {test_df['id'].dtype}")
print(f"Sample IDs: {test_df['id'].values[:10]}")

if os.path.exists(sample_sub_path):
    sample_sub = pd.read_csv(sample_sub_path)
    print(f"\nsample_submission.csv:\n{sample_sub.head()}")

# --- Noise files count ---
if NOISE_DIR:
    noise_files = [f for f in os.listdir(NOISE_DIR) if f.endswith('.wav')]
    print(f"\nNoise files (ESC-50): {len(noise_files)}")

BASE_DIR:   /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup
STEMS_DIR:  /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems
MASHUPS_DIR:/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups
NOISE_DIR:  /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio

TRAINING DATA (Stems per Genre)
  blues       : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  classical   : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  country     : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  disco       : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  hiphop      : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  jazz        : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  metal       : 100 songs | stems: ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
  pop         : 100 songs | stems: ['drums.wav

## Milestone 2: Classical ML Baseline (XGBoost with Audio Features)

# Cell 7 -- Audio Mixing and Feature Extraction for XGBoost (Milestone 2)

This cell prepares the data for our traditional ML model (XGBoost).

**Step 1: Stem Mixing (`mix_stems_to_audio`)**
For XGBoost, we mix the 4 stems (vocals, drums, bass, other) from the **same song** by simply adding them together. Note: this is different from the Deep Learning pipeline (Cells 10-11) where we use Cross-Song Mixing from different songs.
- With 50% probability, we also add background noise from the ESC-50 dataset at a random SNR (Signal-to-Noise Ratio) between 5 and 20 dB.
- The mixed audio is normalized to the [-1, 1] range.

**Step 2: Feature Extraction (`extract_features_from_array`)**
From each mixed audio clip, we extract a dense 1D feature vector using `librosa`:
- **20 MFCCs** (mean and std of each = 40 features): Captures the timbral characteristics of audio.
- **12 Chroma bins** (mean): Captures pitch/note information.
- **Spectral Contrast, Centroid, Bandwidth, Rolloff** (mean each): Captures the brightness and frequency distribution.
- **Zero-Crossing Rate (ZCR)** (mean): A measure of how often the signal crosses zero, useful for distinguishing percussive vs. harmonic sounds.
- **Tempo**: The estimated beats per minute (BPM).

**Key differences between ML and DL preprocessing:**
1. XGBoost uses same-song stem mixing; DL models use cross-song stem mixing with random gain (0.7-1.3).
2. XGBoost noise injection probability is 50%; DL models use 70%.
3. XGBoost uses handcrafted 1D features via `librosa`; CNN/CRNN uses 2D Mel Spectrograms via `torchaudio`; HuBERT uses raw 1D waveform directly.


# Cell 7 -- Audio Mixing and Feature Extraction for XGBoost (Milestone 2)

This cell prepares the data for our traditional ML model (XGBoost).

**Step 1: Stem Mixing (`mix_stems_to_audio`)**
For XGBoost, we mix the 4 stems (vocals, drums, bass, other) from the **same song** by simply adding them together. Note: this is different from the Deep Learning pipeline (Cells 10-11) where we use Cross-Song Mixing from different songs.
- With 50% probability, we also add background noise from the ESC-50 dataset at a random SNR (Signal-to-Noise Ratio) between 5 and 20 dB.
- The mixed audio is normalized to the [-1, 1] range.

**Step 2: Feature Extraction (`extract_features_from_array`)**
From each mixed audio clip, we extract a dense 1D feature vector using `librosa`:
- **20 MFCCs** (mean and std of each = 40 features): Captures the timbral characteristics of audio.
- **12 Chroma bins** (mean): Captures pitch/note information.
- **Spectral Contrast, Centroid, Bandwidth, Rolloff** (mean each): Captures the brightness and frequency distribution.
- **Zero-Crossing Rate (ZCR)** (mean): A measure of how often the signal crosses zero, useful for distinguishing percussive vs. harmonic sounds.
- **Tempo**: The estimated beats per minute (BPM).

**Key differences between ML and DL preprocessing:**
1. XGBoost uses same-song stem mixing; DL models use cross-song stem mixing with random gain (0.7-1.3).
2. XGBoost noise injection probability is 50%; DL models use 70%.
3. XGBoost uses handcrafted 1D features via `librosa`; CNN/CRNN uses 2D Mel Spectrograms via `torchaudio`; HuBERT uses raw 1D waveform directly.


In [7]:
# ============================================================
# Cell 7: Feature Extraction for ML (MFCC + Chroma + Spectral)
# Runs on CPU — no GPU needed
# ============================================================

def mix_stems_to_audio(song_path, sr=SAMPLE_RATE, duration=DURATION):
    """Load and mix all 4 stems from a song folder into one audio array."""
    num_frames = int(sr * duration)
    mixed = np.zeros(num_frames)
    for stem in ['vocals.wav', 'drums.wav', 'bass.wav', 'other.wav']:
        stem_path = os.path.join(song_path, stem)
        if os.path.exists(stem_path):
            y, _ = librosa.load(stem_path, sr=sr, duration=duration)
            if len(y) < num_frames:
                y = np.pad(y, (0, num_frames - len(y)))
            else:
                y = y[:num_frames]
            mixed += y
    max_val = np.max(np.abs(mixed))
    if max_val > 0:
        mixed = mixed / max_val
    return mixed


def extract_features_from_array(y, sr=SAMPLE_RATE):
    """Extract a rich feature vector from a numpy audio array."""
    features = []
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    features.extend(np.mean(mfcc, axis=1))
    features.extend(np.std(mfcc, axis=1))
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features.extend(np.mean(chroma, axis=1))
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    features.extend(np.mean(contrast, axis=1))
    features.append(float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))))
    features.append(float(np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))))
    features.append(float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))))
    features.append(float(np.mean(librosa.feature.zero_crossing_rate(y=y))))
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    features.append(float(np.asarray(tempo).item()))
    return np.array(features)


noise_files_list = []
if NOISE_DIR:
    noise_files_list = [os.path.join(NOISE_DIR, f) for f in os.listdir(NOISE_DIR) if f.endswith('.wav')]

print(f"Extracting features from training data (duration={DURATION}s)...")
X_train_ml = []
y_train_ml = []

for genre in tqdm(genres, desc="Genres"):
    genre_path = os.path.join(STEMS_DIR, genre)
    if not os.path.exists(genre_path):
        continue
    for song_folder in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song_folder)
        mixed = mix_stems_to_audio(song_path)

        if noise_files_list and random.random() < 0.5:
            noise_file = random.choice(noise_files_list)
            noise_y, _ = librosa.load(noise_file, sr=SAMPLE_RATE, duration=DURATION)
            num_frames = len(mixed)
            if len(noise_y) < num_frames:
                noise_y = np.pad(noise_y, (0, num_frames - len(noise_y)))
            else:
                noise_y = noise_y[:num_frames]
            signal_power = np.mean(mixed ** 2) + 1e-10
            noise_power = np.mean(noise_y ** 2) + 1e-10
            snr_db = random.uniform(5, 20)
            scale = np.sqrt(signal_power / (10 ** (snr_db / 10) * noise_power))
            mixed = mixed + scale * noise_y
            max_val = np.max(np.abs(mixed))
            if max_val > 0:
                mixed = mixed / max_val

        feat = extract_features_from_array(mixed, SAMPLE_RATE)
        X_train_ml.append(feat)
        y_train_ml.append(genre)

X_train_ml = np.array(X_train_ml)
print(f"\nFeature matrix shape: {X_train_ml.shape}")
print(f"Total samples: {len(y_train_ml)}")

Extracting features from training data (duration=10s)...


Genres: 100%|██████████| 10/10 [05:50<00:00, 35.00s/it]


Feature matrix shape: (1000, 64)
Total samples: 1000


# Cell 8 -- XGBoost Training and Evaluation (Milestone 2)

Here we train an XGBoost classifier on the handcrafted features extracted in Cell 7.

**Key details:**
- We split the data 85/15 into training and validation sets using `train_test_split` with `stratify` to maintain class balance.
- XGBoost is a gradient-boosted tree algorithm that handles tabular/feature-based data very well.
- We evaluate using **Macro F1 Score** (the competition metric) and **Accuracy**.
- Results are logged to WandB. The XGBoost submission is saved as `submission_xgb.csv`.


# Cell 8 -- XGBoost Training and Evaluation (Milestone 2)

Here we train an XGBoost classifier on the handcrafted features extracted in Cell 7.

**Key details:**
- We split the data 85/15 into training and validation sets using `train_test_split` with `stratify` to maintain class balance.
- XGBoost is a gradient-boosted tree algorithm that handles tabular/feature-based data very well.
- We evaluate using **Macro F1 Score** (the competition metric) and **Accuracy**.
- Results are logged to WandB. The XGBoost submission is saved as `submission_xgb.csv`.


In [8]:
# ============================================================
# Cell 8: Train XGBoost + WandB (Milestone 2)
# WandB LOGIN happens here — first and only place
# ============================================================

# --- WandB Login ---
# New wandb_v1_ keys are longer than 40 chars, so we set via env var
# to bypass the old key-length validation in wandb.login(key=...).
WANDB_KEY = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    WANDB_KEY = user_secrets.get_secret("WANDB_API_KEY")
    print("Got WandB key from Kaggle Secrets.")
except Exception:
    # Hardcoded fallback : not removing this time , but before making it public , i'll reomove or try to make from secreate key link
    WANDB_KEY = "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"
    print("Using hardcoded WandB key.")

if WANDB_KEY:
    try:
        os.environ["WANDB_API_KEY"] = WANDB_KEY
        wandb.login()
        print("WandB logged in successfully.")
    except Exception as e:
        print(f"WandB login failed: {e}")
        print("Continuing in offline mode.")
        os.environ["WANDB_MODE"] = "offline"
else:
    print("No WandB key found. Running in offline mode.")
    os.environ["WANDB_MODE"] = "offline"

wandb.init(project="22f1001611-t12026", name="xgboost-mfcc-baseline")

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y_train_ml)

# Train/val split (80/20, stratified)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_ml, y_encoded, test_size=0.2, random_state=SEED, stratify=y_encoded
)

# Train XGBoost
print("Training XGBoost...")
clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=SEED,
    use_label_encoder=False,
    eval_metric='mlogloss'
)
clf.fit(X_tr, y_tr)

# Evaluate
y_pred = clf.predict(X_val)
val_acc = accuracy_score(y_val, y_pred)
val_f1 = f1_score(y_val, y_pred, average='macro')

print(f"\nXGBoost Results:")
print(f"  Validation Accuracy:  {val_acc:.4f}")
print(f"  Validation Macro F1:  {val_f1:.4f}")
print(f"\nPer-class report:")
print(classification_report(y_val, y_pred, target_names=genres))

# Log to WandB
wandb.log({
    "model": "XGBoost",
    "val_accuracy": val_acc,
    "val_macro_f1": val_f1,
    "n_features": X_train_ml.shape[1],
    "n_samples": len(y_train_ml)
})

# Store for later comparison
xgb_val_acc = val_acc
xgb_val_f1 = val_f1

wandb.finish()
print("XGBoost WandB run complete.")

wandb: Currently logged in as: 22f1001611 (22f1001611-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using hardcoded WandB key.
WandB logged in successfully.


wandb: setting up run 9gzu988v
wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260309_093619-9gzu988v
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run xgboost-mfcc-baseline
wandb: ⭐️ View project at https://wandb.ai/22f1001611-indian-institute-of-technology-madras/22f1001611-t12026
wandb: 🚀 View run at https://wandb.ai/22f1001611-indian-institute-of-technology-madras/22f1001611-t12026/runs/9gzu988v


Training XGBoost...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [09:36:22] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
wandb: updating run metadata



XGBoost Results:
  Validation Accuracy:  0.6350
  Validation Macro F1:  0.6330

Per-class report:
              precision    recall  f1-score   support

       blues       0.58      0.55      0.56        20
   classical       0.88      0.75      0.81        20
     country       0.58      0.55      0.56        20
       disco       0.52      0.70      0.60        20
      hiphop       0.54      0.65      0.59        20
        jazz       0.77      0.85      0.81        20
       metal       0.80      0.80      0.80        20
         pop       0.62      0.50      0.56        20
      reggae       0.65      0.65      0.65        20
        rock       0.44      0.35      0.39        20

    accuracy                           0.64       200
   macro avg       0.64      0.64      0.63       200
weighted avg       0.64      0.64      0.63       200



wandb: 
wandb: Run history:
wandb:   n_features ▁
wandb:    n_samples ▁
wandb: val_accuracy ▁
wandb: val_macro_f1 ▁
wandb: 
wandb: Run summary:
wandb:        model XGBoost
wandb:   n_features 64
wandb:    n_samples 1000
wandb: val_accuracy 0.635
wandb: val_macro_f1 0.63296
wandb: 
wandb: 🚀 View run xgboost-mfcc-baseline at: https://wandb.ai/22f1001611-indian-institute-of-technology-madras/22f1001611-t12026/runs/9gzu988v
wandb: ⭐️ View project at: https://wandb.ai/22f1001611-indian-institute-of-technology-madras/22f1001611-t12026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260309_093619-9gzu988v/logs


XGBoost WandB run complete.
